In [ ]:
from pathlib import Path
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from rasterio.features import rasterize

import sys
lib_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")
sys.path.append(str(lib_dir))
import Robyn_paper_2_defs
import sys, pathlib, importlib
sys.path.append(str(pathlib.Path("../../robyns_libraries").resolve()))
import Robyn_river_floods; importlib.reload(Robyn_river_floods)

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

In [ ]:
catchments_unionized_final = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
# print("Catchments:", len(catchments))

In [ ]:
damage_reduction_min = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_min.tif"
damage_reduction_max = base_path / "dphil_paper_2/processed_data/nbs_river_catchment/damage_reduction/damage_reduction_max.tif"

# TOTAL AREA COST

In [ ]:
# === Priority area (km² & ha) from damage_reduction_max raster ===============

PRIORITY_RASTER = damage_reduction_min  # Path(...) already set earlier
TARGET_CRS = "EPSG:3448"                # metres (Jamaica)

# ---- Priority definition -----------------------------------------------------
MODE = "positive"         # options: "positive", "above_threshold", "top_fraction"
THRESHOLD = 0.0           # used if MODE == "above_threshold"
TOP_FRACTION = 0.10       # used if MODE == "top_fraction"



In [ ]:
def _priority_mask(arr, mode, threshold, top_fraction):
    valid = (~arr.mask) & np.isfinite(arr)
    if mode == "positive":
        return valid & (arr > 0)
    elif mode == "above_threshold":
        return valid & (arr > threshold)
    elif mode == "top_fraction":
        vals = arr[valid].astype("float64")
        if vals.size == 0:
            return np.zeros(arr.shape, dtype=bool)
        q = np.quantile(vals, 1.0 - top_fraction)  # threshold for top X%
        return valid & (arr >= q)
    else:
        raise ValueError(f"Unknown MODE: {mode}")

def priority_area_km2(raster_path: Path, mode=MODE, threshold=THRESHOLD,
                      top_fraction=TOP_FRACTION, target_crs=TARGET_CRS):
    """
    Returns a dict including:
      - area_m2, area_km2, area_ha
      - pixel_area_m2, pixel_area_ha
      - priority_pixels, CRS info
    """
    with rasterio.open(raster_path) as src:
        data = src.read(1, masked=True)
        mask_src = _priority_mask(data, mode, threshold, top_fraction)

        def pack_result(projected_flag, crs_str, px_w, px_h, pix_count, reprojected_to=None):
            px_area_m2 = px_w * px_h
            total_area_m2 = float(pix_count) * px_area_m2
            return {
                "projected": projected_flag,
                "crs": crs_str,
                **({"reprojected_to": reprojected_to} if reprojected_to else {}),
                "pixel_size_m": (px_w, px_h),
                "pixel_area_m2": float(px_area_m2),
                "pixel_area_ha": float(px_area_m2 / 1e4),
                "priority_pixels": int(pix_count),
                "area_m2": float(total_area_m2),
                "area_ha": float(total_area_m2 / 1e4),
                "area_km2": float(total_area_m2 / 1e6),
            }

        if src.crs and src.crs.is_projected:
            px_w = abs(src.transform.a)
            px_h = abs(src.transform.e)
            pix_count = int(np.count_nonzero(mask_src))
            return pack_result(True, str(src.crs), px_w, px_h, pix_count)

        # Reproject boolean mask to metre CRS (nearest-neighbour)
        dst_transform, dst_w, dst_h = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )
        dst = np.zeros((dst_h, dst_w), dtype=np.uint8)
        reproject(
            source=mask_src.astype(np.uint8),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest,
        )
        px_w = abs(dst_transform.a)
        px_h = abs(dst_transform.e)
        pix_count = int((dst == 1).sum())
        return pack_result(False, str(src.crs), px_w, px_h, pix_count, reprojected_to=target_crs)


In [ ]:
# ---- Run it -----------------------------------------------------------------
stats = priority_area_km2(PRIORITY_RASTER)
for k, v in stats.items():
    if k in {"area_m2", "pixel_area_m2"}:
        print(f"{k}: {v:,.2f}")
    elif k in {"area_ha", "pixel_area_ha"}:
        print(f"{k}: {v:,.2f}")
    elif k == "area_km2":
        print(f"{k}: {v:,.3f}")
    else:
        print(f"{k}: {v}")

In [ ]:
land_use = gpd.read_file(base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp")
print(land_use.crs)


In [ ]:
land_use["area_m2"] = land_use.geometry.area
land_use["area_ha"] = land_use["area_m2"] / 1e4
print(f"Total area: {land_use['area_ha'].sum():,.2f} ha")

In [ ]:
cost_per_ha_J = 215_000
total_area_ha = stats["area_ha"]
total_cost_J = total_area_ha * cost_per_ha_J
print(f"Priority area: {total_area_ha:,.2f} ha")
print(f"Total cost (@{cost_per_ha_J:,.0f}/ha): {total_cost_J:,.0f}")

total_cost_USD = total_cost_J / 150
print(f"Total cost (USD @150 J$/USD): ${total_cost_USD:,.0f}")

In [ ]:
# Totals (both already in hectares)
priority_area_ha   = float(total_area_ha)                 # from your stats dict
total_land_use_ha  = float(land_use["area_ha"].sum())

pct = (priority_area_ha / total_land_use_ha * 100) if total_land_use_ha > 0 else np.nan

print(f"Priority area:      {priority_area_ha:,.2f} ha")
print(f"Land-use total:     {total_land_use_ha:,.2f} ha")
print(f"Share of land-use:  {pct:.2f}%")

In [ ]:
three_year_maintenance_costs_per_ha_J = 522_000
three_year_maintenance_costs_per_ha_USD = three_year_maintenance_costs_per_ha_J / 150
one_year_maintenance_costs_per_ha_J = three_year_maintenance_costs_per_ha_J / 3   # 174000.0
one_year_maintenance_costs_per_ha_USD = one_year_maintenance_costs_per_ha_J / 150
one_year_maintenance_costs_per_ha_USD
three_year_maintenance_costs_per_ha_USD

# CATCHMENT LEVEL COSTS

In [ ]:
ID_COL = "catchment_uid"

# Build priority mask on the raster grid (no reprojection)
with rasterio.open(PRIORITY_RASTER) as src:
    data = src.read(1, masked=True)
    priority_mask = _priority_mask(data, MODE, THRESHOLD, TOP_FRACTION).astype(np.uint8)
    transform = src.transform
    height, width = src.height, src.width

# Rasterize catchments to the raster grid
shapes = [(geom, int(cid)) for geom, cid in zip(catchments.geometry, catchments[ID_COL])]
# change just this in your rasterize() call:
id_grid = rasterize(
    shapes=shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="int32",
    all_touched=True,  # <--- was False
)

# Count priority pixels per catchment
valid = (priority_mask == 1) & (id_grid > 0)
ids = id_grid[valid]

vc = pd.Series(ids, dtype="int32").value_counts().sort_index()
per_catchment = (vc.rename("priority_pixels")
                   .to_frame()
                   .reset_index()
                   .rename(columns={"index": ID_COL}))

# Convert to areas
px_w = abs(transform.a)
px_h = abs(transform.e)
px_area_m2 = px_w * px_h
per_catchment["area_m2"] = per_catchment["priority_pixels"] * px_area_m2
per_catchment["area_ha"] = per_catchment["area_m2"] / 1e4
per_catchment["area_km2"] = per_catchment["area_m2"] / 1e6

# Costs (as requested)
per_catchment["costs_J"] = per_catchment["area_ha"] * 215_000
per_catchment["costs_USD"] = per_catchment["costs_J"] / 150

# Maintenance (USD) as requested
per_catchment["one_year_maintenance_costs_USD"] = (
    per_catchment["area_ha"] * one_year_maintenance_costs_per_ha_USD
)

# Maintenance (USD) as requested
per_catchment["three_year_maintenance_costs_USD"] = (
    per_catchment["area_ha"] * three_year_maintenance_costs_per_ha_USD
)

per_catchment.head()

In [ ]:
# # Sum at catchment level
sum_costs_usd = float(per_catchment["costs_USD"].sum(skipna=True))
sum_one_year_maintenance_costs_usd = float(per_catchment["one_year_maintenance_costs_USD"].sum(skipna=True))
sum_three_year_maintenance_costs_usd = float(per_catchment["three_year_maintenance_costs_USD"].sum(skipna=True))
sum_area_ha   = float(per_catchment["area_ha"].sum(skipna=True))

In [ ]:
# --- Apples-to-apples check: total priority INSIDE catchments ----------------
# Build a single boolean mask of catchments on the SAME raster grid
catchments_mask = rasterize(
    shapes=[(geom, 1) for geom in catchments.geometry],
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8",
    all_touched=True,   # match your per-catchment choice
)

# Total priority pixels confined to catchments
priority_within = (priority_mask == 1) & (catchments_mask == 1)
total_within_pixels = int(priority_within.sum())

# Convert to area and cost using your existing px_area_m2 and rates
total_within_ha  = total_within_pixels * px_area_m2 / 1e4
total_within_usd = (total_within_ha * 215_000) / 150

print(f"Priority area within catchments: {total_within_ha:,.2f} ha")
print(f"Whole-island (within-catchments) cost: ${total_within_usd:,.0f} USD")
print(f"Sum of per-catchment costs:            ${sum_costs_usd:,.0f} USD")
print(f"Sum of per-catchment one year maintenance costs:            ${sum_one_year_maintenance_costs_usd:,.0f} USD")
print(f"Sum of per-catchment three year maintenance costs:            ${sum_three_year_maintenance_costs_usd:,.0f} USD")
print(f"Difference:                            ${sum_costs_usd - total_within_usd:,.0f} USD")

# Tolerance ~ a few pixels worth (use exactly one-pixel USD as a yardstick)
one_pixel_usd = (px_area_m2 / 1e4) * 215_000 / 150
assert np.isclose(sum_costs_usd, total_within_usd, atol=5*one_pixel_usd), "Per-catchment sum ≠ within-catchments total."

In [ ]:
# --- Ensure all catchments are present; fill zeros ----------------------------
per_catchment_full = (
    catchments[[ID_COL]].drop_duplicates()
    .merge(per_catchment, on=ID_COL, how="left")
)

num_cols = ["priority_pixels", "area_m2", "area_ha", "area_km2", "costs_J", "costs_USD"]
for c in num_cols:
    if c not in per_catchment_full.columns:
        per_catchment_full[c] = 0.0
per_catchment_full[num_cols] = per_catchment_full[num_cols].fillna(0.0)

# --- Sanity check: per-catchment sum vs the within-catchments total ----------
sum_costs_usd_full = float(per_catchment_full["costs_USD"].sum())
sum_three_year_maintenance_costs_usd_full = float(per_catchment_full["three_year_maintenance_costs_USD"].sum())

sum_area_ha_full   = float(per_catchment_full["area_ha"].sum())

print(f"Sum capital costs (per-catchment): ${sum_costs_usd_full:,.0f} USD")
print(f"Sum three year maintenance costs (per-catchment): ${sum_three_year_maintenance_costs_usd_full:,.0f} USD")

print(f"Sum area  (per-catchment): {sum_area_ha_full:,.2f} ha")

# Compare to your 'within-catchments' total from the apples-to-apples check
print(f"Whole-island (within-catchments) cost: ${total_within_usd:,.0f} USD")
print(f"Difference:                            ${sum_costs_usd_full - total_within_usd:,.0f} USD")

# Tight equality (allowing tiny floating noise)
import numpy as np
assert np.isclose(sum_costs_usd_full, total_within_usd, atol=1e-6), "Per-catchment sum != within-catchments total."

# --- (Optional) quick view: top 10 most costly catchments --------------------
print(
    per_catchment_full[[ID_COL, "area_ha", "costs_J", "costs_USD","one_year_maintenance_costs_USD", "three_year_maintenance_costs_USD"]]
      .sort_values("costs_USD", ascending=False)
      .head(10)
      .to_string(index=False, formatters={
          "area_ha":   lambda x: f"{x:,.2f}",
          "costs_J":   lambda x: f"{x:,.0f}",
          "costs_USD": lambda x: f"{x:,.0f}",
          "one_year_maintenance_costs_USD": lambda x: f"{x:,.0f}",
          "three_year_maintenance_costs_USD": lambda x: f"{x:,.0f}",

      })
)

(per_catchment_full[[ID_COL, "area_ha", "costs_J", "costs_USD", "one_year_maintenance_costs_USD", "three_year_maintenance_costs_USD"]]
  .sort_values("costs_USD", ascending=False)
  .to_csv(base_path / "dphil_paper_2/results/restoration_costs/catchment_priority_costs_min.csv", index=False))